# Stage A — Neural Galactic Potential (Proof of Concept)
## Physics-Informed SIREN for Dark Matter Halo Modelling

**Run order:** Execute every cell top to bottom in sequence.
Each cell is a self-contained step. If a step fails, fix it before moving on.

**Runtime:** ~5-10 minutes on a T4 GPU (free Colab). Enable GPU: Runtime → Change runtime type → T4 GPU.

---
## Cell 1 — Environment Setup

In [ ]:
# Install any missing packages (torch is pre-installed in Colab)
!pip install torch numpy scipy matplotlib --quiet

import torch
import numpy as np
import matplotlib.pyplot as plt
import os, sys

print(f'PyTorch    : {torch.__version__}')
print(f'NumPy      : {np.__version__}')
print(f'CUDA       : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU        : {torch.cuda.get_device_name(0)}')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using      : {DEVICE}')

---
## Cell 2 — Upload Project Files
Upload the entire `siren_grav` folder to Colab, then set the path.

In [ ]:
# Option A: Upload files via Colab file browser (left sidebar → upload)
# Then set this path to where you uploaded:
PROJECT_PATH = '/content/siren_grav'
sys.path.insert(0, PROJECT_PATH)

# Verify structure
for f in ['src/nfw.py', 'src/siren.py', 'src/physics.py',
          'src/dataset.py', 'src/trainer.py', 'experiments/run_stage_a.py']:
    exists = os.path.exists(os.path.join(PROJECT_PATH, f))
    print(f'  {f}: {"OK" if exists else "MISSING"}')

---
## Cell 3 — Gate Test 1: NFW Analytical Functions

In [ ]:
os.chdir(PROJECT_PATH)
!python tests/test_nfw.py

---
## Cell 4 — Gate Test 2: SIREN Initialization

In [ ]:
!python tests/test_siren_init.py

---
## Cell 5 — Gate Test 3: Laplacian Cross-Verification

In [ ]:
!python tests/test_laplacian.py

---
## Cell 6 — Full Stage A Experiment
Trains all three models, evaluates, generates benchmark table and all plots.
~5-10 min on GPU.

In [ ]:
!python experiments/run_stage_a.py

---
## Cell 7 — Display Results

In [ ]:
OUTPUT_DIR = os.path.join(PROJECT_PATH, 'outputs/stage_a')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_files = ['training_curves.png', 'density_comparison.png', 'radial_profile.png']

for ax, fname in zip(axes, plot_files):
    path = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(path):
        img = plt.imread(path)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(fname.replace('.png','').replace('_',' ').title())
    else:
        ax.text(0.5, 0.5, f'{fname}\nnot found', ha='center', va='center')

plt.tight_layout()
plt.show()

# Display Poisson residual map
poisson_path = os.path.join(OUTPUT_DIR, 'poisson_residual_map.png')
if os.path.exists(poisson_path):
    img = plt.imread(poisson_path)
    plt.figure(figsize=(7, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Poisson Residual Map')
    plt.show()

---
## Cell 8 — Read Benchmark Results

In [ ]:
import json

results_path = os.path.join(OUTPUT_DIR, 'benchmark_results.json')
with open(results_path) as f:
    results = json.load(f)

print(f'{"Model":<28} {"Rho err%":>10} {"Phi err%":>10} {"Poisson%":>10} {"Size KB":>10}')
print('-' * 65)
for name, m in results.items():
    print(f'{name:<28} {m["rho_rel_error"]:>9.2f}% '
          f'{m["phi_rel_error"]:>9.2f}% '
          f'{m["poisson_residual"]:>9.2f}% '
          f'{m["weight_size_kb"]:>9.1f}')

---
## Cell 9 — Stage A Verdict
Stage A is complete when:
- All 3 gate tests pass
- SIREN (data only) Rho error < 1%
- SIREN + Poisson residual < 5%
- Model file size < 1 MB vs ~50 MB raw data

In [ ]:
print('STAGE A VERDICT')
print('=' * 50)
for name, m in results.items():
    rho_ok     = m['rho_rel_error']    < 5.0
    phi_ok     = m['phi_rel_error']    < 5.0
    poisson_ok = m['poisson_residual'] < 10.0  # PINN model
    size_ok    = m['weight_size_kb']   < 1500
    print(f'\n{name}:')
    print(f'  Rho err < 5%    : {"PASS" if rho_ok else "FAIL"}  ({m["rho_rel_error"]:.2f}%)')
    print(f'  Phi err < 5%    : {"PASS" if phi_ok else "FAIL"}  ({m["phi_rel_error"]:.2f}%)')
    if 'PINN' in name:
        print(f'  Poisson < 10%   : {"PASS" if poisson_ok else "FAIL"}  ({m["poisson_residual"]:.2f}%)')
    print(f'  Size < 1.5 MB   : {"PASS" if size_ok else "FAIL"}  ({m["weight_size_kb"]/1024:.2f} MB)')